<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/SPARQL%20for%20Graph%20Querying/09b-lab-sparql-by-hand-three-graphs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9b — Reading RDF Connections, Then Writing SPARQL

Most people who get stuck writing SPARQL are not stuck on the syntax. They are stuck because they **cannot see how the triples connect to each other** — so they don't know what path to ask for. This lab fixes that first, using only the simplest queries: **follow arrows and read off labels.** No filters, no counting, no special keywords — just matching connections.

The whole idea in one sentence:

> **RDF is a graph of connections. To answer a question you trace a path of connections by hand. A SPARQL query is just that path, written down.**

So every exercise runs the same loop, and step 2 is the one that matters most:

1. **Read the question.**
2. **Trace the connections by hand.** Find the starting node, follow the arrows (each `subject predicate object` triple is one arrow), and note which node you land on. *Write the path down as nodes and arrows before anything else.*
3. **Read off your answer** from where the path lands.
4. **Turn the path into SPARQL** — one arrow you followed becomes one triple pattern — run it, and check it matches your hand answer.

There is a collapsed **Solution** under every question showing the traced path, the hand answer, and the query. Don't open it until you've drawn your own path.

**Three graphs, getting harder only by *how many arrows you follow* — and each one small enough to fit on a single slide and work out by hand:**

| Part | Graph | Size | The connection skill you build |
|---|---|---|---|
| A | A tiny **bookshop** | 3 books | one arrow; then *two arrows through a shared node* (the join, to reach a name) |
| B | A tiny **movie** graph | 3 films | the same, where the URIs are **opaque IDs** so you *must* follow an arrow to a label |
| C | A **family tree** | 7 people | following the *same* arrow again and again — chains of connections |

---

## First: how to read RDF as connections

An RDF file is a list of **triples**. Every triple is one arrow in a graph:

```
subject  ─predicate→  object
```

Read `ex:b1 ex:title "Clean Code"` as: *the node `ex:b1` has an arrow labelled `ex:title` pointing to `"Clean Code"`.*

Two kinds of object sit at the end of those arrows, and telling them apart is the key skill:

- A **literal** (in quotes) — `"Clean Code"`. This is a dead end: a value, nothing points onward from it.
- A **URI** — `ex:b1`, `ex:martin`. This is a **node you can keep travelling from**, because it may be the *subject* of other triples.

**Nodes connect when the same URI appears in two triples.** Look at these two lines:

```
ex:b1     ex:author  ex:martin .       ← ex:martin is the OBJECT here
ex:martin ex:name    "Robert Martin" . ← ex:martin is the SUBJECT here
```

`ex:martin` is the landing point of the first arrow and the launch point of the second. That shared node is a **connection**: it lets you travel `ex:b1 → ex:martin → "Robert Martin"` and discover that book `ex:b1`'s author is named Robert Martin — even though no single triple says so.

**This is exactly what a SPARQL join is.** When you write

```sparql
?b ex:author ?author .
?author ex:name ?name .
```

the repeated `?author` is you telling the engine: *the node I land on in line 1 is the same node I leave from in line 2* — the same hand-off you did by finger. Every join you will write in this lab is just "the same node, reached from two sides."

Keep this table next to you:

| In the RDF you see… | It means… | When you trace / query… |
|---|---|---|
| `A pred B` | an arrow `A ─pred→ B` | follow it forward: from `A` you reach `B` |
| the same URI as object here and subject there | a connection between two arrows | hand off from one arrow to the next through that node |
| a literal object (in quotes) | a value, a dead end | you've arrived — read it off |
| a variable `?x` reused in two patterns | "must be the same node" | that's the join; it welds the two arrows together |

Now let's read some graphs.

## Setup

Run once. Installs the `%%sparql` cell magic (from the course's `cellspell` package) and `rdflib`. Each query cell starts with `%%sparql --file <name>.ttl` and runs against that Turtle file.

In [ ]:
# ── Install the SPARQL cell magic + rdflib ──────────────
!pip install "cellspell[sparql] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
!pip install -q rdflib
%load_ext cellspell.sparql

---

## Part A — The Bookshop  🟢 *one arrow, then two*

Three books — the whole graph fits on a slide. Read it as arrows. A book node has arrows for `ex:title` and `ex:topic` (both landing on **literals** — dead ends), and one `ex:author` arrow landing on a **person URI** — a node you can travel onward from to reach the author's name.

Run the next cell to write `bookshop.ttl`, then read it as a graph, not as text.

In [ ]:
%%writefile bookshop.ttl
@prefix ex:  <http://example.org/book#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:b1 rdf:type ex:Book ; ex:title "Clean Code" ;
      ex:author ex:martin ; ex:topic "Software" .

ex:b2 rdf:type ex:Book ; ex:title "SQL for Smarties" ;
      ex:author ex:celko ; ex:topic "Databases" .

ex:b3 rdf:type ex:Book ; ex:title "Designing Data-Intensive Applications" ;
      ex:author ex:kleppmann ; ex:topic "Databases" .

ex:martin     ex:name "Robert Martin" .
ex:celko      ex:name "Joe Celko" .
ex:kleppmann  ex:name "Martin Kleppmann" .

**Warm-up — draw the connections (no SPARQL yet).** For book `ex:b1`, list every arrow leaving it and say where each lands. Which arrows end at a dead-end literal, and which one lands on a node you can keep travelling from?

<details>
<summary>Check yourself</summary>

`ex:b1` has four arrows out:

```
ex:b1 ─rdf:type→ ex:Book
ex:b1 ─ex:title→ "Clean Code"      (literal — dead end)
ex:b1 ─ex:author→ ex:martin        (URI — travel onward!)
ex:b1 ─ex:topic→ "Software"        (literal — dead end)
```

Only `ex:author` lands on a node (`ex:martin`) with its own outgoing arrow: `ex:martin ─ex:name→ "Robert Martin"`. So to get the author's *name* from a book you must make **two** hops, handing off through `ex:martin`. Every other fact is one hop away.

</details>

**Standard prefix for Part A** — paste at the top of each query cell:

```sparql
PREFIX ex:  <http://example.org/book#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
```

### A1 — List every book title  *(follow one arrow)*

**Trace the connections.** From each book node, follow the single `ex:title` arrow. Where does it land? How many books are there, so how many arrows, so how many rows?

**Now write it.** One arrow you followed = one triple pattern. Bind the book with `?b` and the landing literal with `?title`.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

# One arrow: (some book) --ex:title--> ?title
SELECT ?title WHERE {
    # your triple pattern here
}

<details>
<summary>Solution</summary>

**Path traced:** `?b ─ex:title→ ?title`, for every book node. Three book nodes → three arrows → **3 rows**: *Clean Code, SQL for Smarties, Designing Data-Intensive Applications.*

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title WHERE {
    ?b ex:title ?title .
}
```

The single pattern *is* the single arrow. `?b` isn't in the SELECT but it still does the work: it insists `?title` be reached *from a book node*, not from anywhere.

</details>

### A2 — Titles of the Databases books  *(match a fixed value)*

**Trace the connections.** Keep only book nodes whose `ex:topic` arrow lands on the literal `"Databases"`. From each survivor, follow its `ex:title` arrow and read the title.

**Now write it.** Two arrows leaving the *same* book node: one pinned to the value `"Databases"`, one binding `?title`.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

SELECT ?title WHERE {
    # same book node: --ex:topic--> "Databases"  AND  --ex:title--> ?title
}

<details>
<summary>Solution</summary>

**Path traced:** the nodes with an `ex:topic` arrow to `"Databases"` are `ex:b2` and `ex:b3`. Follow each one's `ex:title` → **2 rows**: *SQL for Smarties, Designing Data-Intensive Applications.*

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title WHERE {
    ?b ex:topic "Databases" ;
       ex:title ?title .
}
```

Putting the literal `"Databases"` directly in the pattern (instead of a variable) is how you **search by value** — only books connected to that exact topic survive. The `;` keeps both arrows on the same subject `?b`, so the title you read belongs to that same book.

</details>

### A3 — Each title with its author's name  *(the two-arrow hand-off = the join)*

This is the one from the primer. The author's **name** is not on the book. You go: book `─ex:author→` person node `─ex:name→` name. The person node is the hand-off.

**Trace the connections.** Take `ex:b1`: follow `ex:author` → you land on `ex:martin`. Now *leave* `ex:martin` along `ex:name` → `"Robert Martin"`. Do that for all three books and write the *(title, name)* pairs.

**Now write it.** Two patterns. The landing node of arrow 1 is the launch node of arrow 2 — so **use the same variable** `?author` in both. That shared variable *is* the connection.

In [ ]:
%%sparql --file bookshop.ttl
PREFIX ex:  <http://example.org/book#>

SELECT ?title ?name WHERE {
    # arrow 1: ?b --ex:title--> ?title  and  ?b --ex:author--> ?author
    # arrow 2: ?author --ex:name--> ?name       (SAME ?author = the hand-off)
}

<details>
<summary>Solution</summary>

**Path traced** (for each book): `?b ─ex:author→ ?author ─ex:name→ ?name`, and separately `?b ─ex:title→ ?title`. Three books, each with one author, each author with one name → **3 rows**:

| ?title | ?name |
|---|---|
| Clean Code | Robert Martin |
| SQL for Smarties | Joe Celko |
| Designing Data-Intensive Applications | Martin Kleppmann |

**Query:**

```sparql
PREFIX ex: <http://example.org/book#>
SELECT ?title ?name WHERE {
    ?b ex:title ?title ;
       ex:author ?author .
    ?author ex:name ?name .
}
```

`?author` appears as the **object** of `ex:author` and the **subject** of `ex:name` — exactly the "same URI in two triples" pattern from the primer. Repeating the variable tells the engine *these two arrows meet at the same node*. That is the entire mechanism of a join; everything else in this lab is more of this.

</details>

---

## Part B — The Movie Graph  🟡 *why you always end on a label*

Same moves as Part A, but now the graph is built the way real knowledge graphs are: the nodes have **opaque IDs** — `ex:f1`, `ex:p1`, `ex:g1` — that tell you nothing on their own. `ex:f1` could be a film, a person, anything. The *only* way to get a human-readable answer is to follow one more arrow to a **label** (`ex:title`, `ex:name`, `ex:label`). So label extraction stops being optional here — it's how you finish every query.

The whole graph still fits on a slide. Read it and notice: films link to a director and a genre **by ID**, and the readable names live one arrow further out.

In [ ]:
%%writefile movies.ttl
@prefix ex:  <http://example.org/movie#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

# Films — identified by opaque IDs, linked to a director and a genre by ID
ex:f1 rdf:type ex:Film ; ex:title "Inception"    ; ex:director ex:p1 ; ex:genre ex:g1 .
ex:f2 rdf:type ex:Film ; ex:title "Interstellar" ; ex:director ex:p1 ; ex:genre ex:g1 .
ex:f3 rdf:type ex:Film ; ex:title "Barbie"       ; ex:director ex:p2 ; ex:genre ex:g2 .

# People — the readable name is one arrow away
ex:p1 ex:name "Christopher Nolan" .
ex:p2 ex:name "Greta Gerwig" .

# Genres — likewise
ex:g1 ex:label "Science Fiction" .
ex:g2 ex:label "Comedy" .

**Warm-up — what is `ex:p1`?** Without following an arrow, can you tell? Now follow its `ex:name` arrow. This is the habit the whole part drills: an ID is just a handle; the meaning is one label-arrow away.

<details>
<summary>Check yourself</summary>

`ex:p1` on its own is opaque. It only becomes meaningful when you follow `ex:p1 ─ex:name→ "Christopher Nolan"`. Two films point their `ex:director` arrow at this same node `ex:p1` — that shared target is what says "these two films have the same director," even before you know his name.

</details>

**Standard prefix for Part B:**

```sparql
PREFIX ex:  <http://example.org/movie#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
```

### B1 — List every film title  *(one arrow, warm-up)*

**Trace the connections.** From each film node, follow `ex:title`. How many rows?

**Now write it.** Just like A1.

In [ ]:
%%sparql --file movies.ttl
PREFIX ex:  <http://example.org/movie#>

SELECT ?title WHERE {
    # (some film) --ex:title--> ?title
}

<details>
<summary>Solution</summary>

**Path traced:** three film nodes, each with one `ex:title` arrow → **3 rows**: *Inception, Interstellar, Barbie.*

**Query:**

```sparql
PREFIX ex: <http://example.org/movie#>
SELECT ?title WHERE {
    ?f ex:title ?title .
}
```

</details>

### B2 — Films directed by `ex:p1`  *(search by ID, extract the label)*

You're told the director's ID is `ex:p1` (that's Nolan — you'd confirm it by following his `ex:name` arrow). Find his films' titles.

**Trace the connections.** Which film nodes have an `ex:director` arrow pointing at `ex:p1`? For each, follow `ex:title`.

**Now write it.** Pin `ex:director` to the value `ex:p1`, then read the title off the same film node.

In [ ]:
%%sparql --file movies.ttl
PREFIX ex:  <http://example.org/movie#>

SELECT ?title WHERE {
    # same film node: --ex:director--> ex:p1  AND  --ex:title--> ?title
}

<details>
<summary>Solution</summary>

**Path traced:** `ex:f1` and `ex:f2` both point `ex:director` at `ex:p1`; `ex:f3` points at `ex:p2` (dropped). Follow the two survivors' `ex:title` → **2 rows**: *Inception, Interstellar.*

**Query:**

```sparql
PREFIX ex: <http://example.org/movie#>
SELECT ?title WHERE {
    ?f ex:director ex:p1 ;
       ex:title ?title .
}
```

Same shape as A2, but the fixed value is a **URI** (`ex:p1`) instead of a literal. Searching by a node's ID is the everyday move in a knowledge graph — you match on the opaque handle, then follow a label arrow to make the result readable.

</details>

### B3 — Every film with its genre name  *(two arrows to a label)*

The genre is stored by ID (`ex:g1`), and the readable genre name is one arrow further along (`ex:g1 ─ex:label→ "Science Fiction"`). So this is a two-hop, just like the author join in A3 — film → genre node → genre label.

**Trace the connections.** For `ex:f1`: `ex:genre` → `ex:g1`, then `ex:label` → `"Science Fiction"`. Do all three films and write the *(title, genre)* pairs.

**Now write it.** Hand off through a shared `?g` variable.

In [ ]:
%%sparql --file movies.ttl
PREFIX ex:  <http://example.org/movie#>

SELECT ?title ?genre WHERE {
    # ?f --ex:title--> ?title  and  ?f --ex:genre--> ?g
    # ?g --ex:label--> ?genre        (SAME ?g = the hand-off)
}

<details>
<summary>Solution</summary>

**Path traced:** f1→g1→"Science Fiction", f2→g1→"Science Fiction", f3→g2→"Comedy" → **3 rows**:

| ?title | ?genre |
|---|---|
| Inception | Science Fiction |
| Interstellar | Science Fiction |
| Barbie | Comedy |

**Query:**

```sparql
PREFIX ex: <http://example.org/movie#>
SELECT ?title ?genre WHERE {
    ?f ex:title ?title ;
       ex:genre ?g .
    ?g ex:label ?genre .
}
```

Identical mechanism to A3's author join — a shared node (`?g`) welds two arrows together. Notice Inception and Interstellar both land on the *same* genre node `ex:g1`; that shared target is why they report the same genre name.

</details>

---

## Part C — The Family Tree  🔴 *chains of the same connection*

One arrow type, `ex:parentOf` (parent → child), repeated to build a tree — seven people, still a single slide. Everything here is about **chaining triples**: each triple hands off to the next through a shared node, exactly like the author join in A3. Child = one triple. Grandchild = two triples chained. Great-grandchild = three. Each question just adds one more triple of the same kind.

The tree (parents above children):

```
        Ada
      ┌──┴───┐
     Cara   Dev
    ┌─┴─┐    │
  Eli  Fay  Gus
   │
  Hana
```

Read every `ex:parentOf` line of the file as one `parent ─parentOf→ child` arrow and check it against this picture.

In [ ]:
%%writefile family.ttl
@prefix ex:   <http://example.org/fam#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:ada  foaf:name "Ada" .   ex:cara foaf:name "Cara" .
ex:dev  foaf:name "Dev" .   ex:eli  foaf:name "Eli" .
ex:fay  foaf:name "Fay" .   ex:gus  foaf:name "Gus" .
ex:hana foaf:name "Hana" .

# ex:parentOf : parent -> child  (one arrow type, used six times)
ex:ada  ex:parentOf ex:cara .   ex:ada  ex:parentOf ex:dev .
ex:cara ex:parentOf ex:eli .    ex:cara ex:parentOf ex:fay .
ex:dev  ex:parentOf ex:gus .
ex:eli  ex:parentOf ex:hana .

**Warm-up — read arrows, rebuild the tree.** Cover the diagram. Using only the six `ex:parentOf` lines, write who each person's children are. Then uncover and check. (This *is* the skill — the diagram is just those six arrows drawn.)

**Standard prefix for Part C:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
```

### C1 — Ada's children  *(one arrow, then the label)*

**Trace the connections.** Follow every `ex:parentOf` arrow leaving `ex:ada`. Land on each child node, then follow `foaf:name` to read the name.

**Now write it.** One `ex:parentOf` hop, then a label hop.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE {
    # ex:ada --ex:parentOf--> ?child ;  ?child --foaf:name--> ?name
}

<details>
<summary>Solution</summary>

**Path traced:** `ex:ada` has two `ex:parentOf` arrows → Cara, Dev → **2 rows**.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ex:ada ex:parentOf ?child .
    ?child foaf:name ?name .
}
```

Starting from a fixed node (`ex:ada`) instead of a variable just means "begin the trace here." Then the same label hop as always turns the child node into a readable name.

</details>

### C2 — Ada's grandchildren  *(chain two triples)*

Grandchild = one `ex:parentOf` triple from Ada to a child, then a second `ex:parentOf` triple from that child. The two triples chain through a shared `?child` node — the same hand-off you used for the author's name in A3.

**Trace the connections.** Ada → {Cara, Dev}. Cara → {Eli, Fay}. Dev → {Gus}. Two hops lands you on…?

**Now write it.** Two `ex:parentOf` triples joined on `?child`, then the label hop. You don't need `?child` in the output — it's just the stepping-stone.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE {
    # ex:ada --ex:parentOf--> ?child       (triple 1)
    # ?child --ex:parentOf--> ?gc           (triple 2, SAME ?child = the hand-off)
    # ?gc --foaf:name--> ?name
}

<details>
<summary>Solution</summary>

**Path traced:** Ada→{Cara,Dev}→{Eli,Fay,Gus} → **3 rows**: Eli, Fay, Gus.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ex:ada ex:parentOf ?child .
    ?child ex:parentOf ?gc .
    ?gc foaf:name ?name .
}
```

`?child` is the shared node: object of triple 1, subject of triple 2 — the two `ex:parentOf` arrows meet there. That's the same join mechanism as A3's author, just with the *same* predicate on both sides. `?gc` (the grandchild) is where the chain lands; the final `foaf:name` triple turns it into a readable name.

</details>

### C3 — Ada's great-grandchildren  *(chain three triples)*

One more triple than C2. Great-grandchild = `ex:parentOf` three times, each triple handing off to the next.

**Trace the connections.** Ada → children → grandchildren → *their* children. Who is three hops below Ada?

**Now write it.** Three `ex:parentOf` triples in a row (`?child`, then `?gc`, then `?ggc`), then the label hop.

In [ ]:
%%sparql --file family.ttl
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?name WHERE {
    # ex:ada --ex:parentOf--> ?child
    # ?child --ex:parentOf--> ?gc
    # ?gc --ex:parentOf--> ?ggc
    # ?ggc --foaf:name--> ?name
}

<details>
<summary>Solution</summary>

**Path traced:** Ada→{Cara,Dev}→{Eli,Fay,Gus}→… only Eli has a child, Hana → **1 row**: Hana.

**Query:**

```sparql
PREFIX ex:   <http://example.org/fam#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name WHERE {
    ex:ada ex:parentOf ?child .
    ?child ex:parentOf ?gc .
    ?gc ex:parentOf ?ggc .
    ?ggc foaf:name ?name .
}
```

Nothing new — you just chained one more `ex:parentOf` triple. "Child", "grandchild", "great-grandchild" aren't different predicates in the data; they're the same `ex:parentOf` connection, chained one, two, or three times. Each extra triple is one more arrow you followed by hand.

</details>

---

## What you actually practised

Not "SPARQL syntax" — **reading a graph as connections and copying the path into a query.** Every query here was some number of arrows followed to a label:

| When you traced… | You wrote… | First seen |
|---|---|---|
| one arrow to a value | one triple pattern | A1, B1, C1 |
| an arrow pinned to a fixed value / ID | a literal or URI in the pattern | A2, B2 |
| a hand-off through a shared node (to reach a name) | a repeated variable — the join | A3, B3 |
| a chain of the same arrow, 2 or 3 times | that many triples, joined nose-to-tail | C2, C3 |

Every query was the same two moves: **follow arrows by matching triples, and hand off through shared variables.** That's it — no special keywords needed.

**The one habit to keep:** before you type a query, draw the path — starting node, each arrow, landing node, and a final arrow to a label if the answer is a name. Then translate it arrow-by-arrow, one triple per arrow. When your hand answer and the engine disagree, re-trace the arrows until you find the connection you misread. That gap is the whole point.

Ready for bigger graphs and richer queries — filtering, optional data, counting, negation, property-path shortcuts, multiple source files, and a live Wikidata endpoint? Continue to **Lab 9 — SPARQL: Querying the Movie Knowledge Graph**.